# 02 — Embeddings

**Input:** `../data/processed/pm_day_clean.csv`
**Output:** `../data/processed/embeddings.npy`, `../data/processed/embedding_rows.csv`

**Description:**
- Load cleaned day-level data
- Compute sentence-transformer embeddings of PM text
- Cache embeddings to disk (reloads if already computed)
- Save a row map so embeddings stay aligned to data

In [2]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_clean.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
ROW_MAP_PATH = os.path.join("..", "data", "processed", "embedding_rows.csv")

EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
PID_COL = "expiwell_id_clean"

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [3]:
# =========================
# LOAD DATA
# =========================
pm_day = pd.read_csv(DATA_PATH)
print("Loaded:", pm_day.shape)
print("Participants:", pm_day[PID_COL].nunique())

Loaded: (2511, 26)
Participants: 123


In [4]:
# =========================
# COMPUTE OR LOAD EMBEDDINGS
# =========================
if os.path.exists(EMBED_PATH):
    X_text = np.load(EMBED_PATH)
    print("Loaded cached embeddings:", X_text.shape)
    if X_text.shape[0] != len(pm_day):
        print("WARNING: Row count mismatch. Recomputing.")
        os.remove(EMBED_PATH)
        X_text = None
else:
    X_text = None

if X_text is None:
    model = SentenceTransformer(EMBED_MODEL)
    X_text = model.encode(
        pm_day["pm_day_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(EMBED_PATH, X_text)
    print("Saved embeddings:", EMBED_PATH)

print("X_text shape:", X_text.shape)

Loaded cached embeddings: (2511, 768)
X_text shape: (2511, 768)


In [5]:
# =========================
# SAVE ROW MAP (for alignment verification)
# =========================
row_map = pm_day[[PID_COL, "ema_date"]].copy()
row_map["embed_row"] = range(len(row_map))
row_map.to_csv(ROW_MAP_PATH, index=False)
print("Saved row map:", ROW_MAP_PATH)

Saved row map: ../data/processed/embedding_rows.csv
